# RAG sobre Historia de las Guerras Mundiales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhonattanreales21/nlp_icesi/blob/main/unidad_6/Cano_Reales_RAG_v3.ipynb)

**Autores:** Andrés Cano & Jhonattan Reales  
**Curso:** Procesamiento de Lenguaje Natural — Maestría MIAA, ICESI  
**Unidad 6:** Retrieval-Augmented Generation (RAG)

---

Objetivo
----

En este taller construimos un **chatbot conversacional basado en RAG** especializado en la historia de las Guerras Mundiales. A diferencia de un LLM estándar, nuestro sistema recupera fragmentos relevantes de una base de conocimiento propia (artículos de Wikipedia en inglés) antes de generar cada respuesta, lo que garantiza mayor precisión y trazabilidad de las fuentes.

Problema a resolver
----

Los modelos de lenguaje por sí solos pueden generar respuestas fluidas, pero no siempre fundamentadas en documentos específicos. Esto puede producir respuestas imprecisas o alucinadas. Para mitigar este problema, se propone una arquitectura RAG, en la que el modelo no responde únicamente desde su conocimiento paramétrico, sino apoyándose en fragmentos de documentos relevantes recuperados dinámicamente.

**¿Por qué RAG en lugar de un LLM estándar?**

1. **Alucinaciones reducidas**: el LLM sólo puede responder con lo que está en el corpus recuperado.
2. **Trazabilidad**: cada respuesta incluye las fuentes (artículos de Wikipedia) que la fundamentan.
3. **Conocimiento actualizable**: basta con re-indexar documentos nuevos sin reentrenar el modelo.
4. **Eficiencia**: usar un modelo pequeño con un buen retriever supera a un modelo grande sin contexto.

Componentes del sistema
----

El sistema fue construido seleccionando herramientas que balancean rendimiento, reproducibilidad y ejecución local, alineadas con el objetivo de crear un chatbot RAG especializado en historia de las Guerras Mundiales.

- **Dataset:** `wikimedia/wikipedia` (filtrado por palabras claves de WW1/WW2) — Artículos de Wikipedia en inglés sobre ambas guerras mundiales, sirviendo como base de conocimiento verificable y trazable para las respuestas del chatbot.
- **LLM:** `phi3:mini` (Ollama) — Modelo de 3.8B parámetros de Microsoft ejecutado localmente, permite ejecución rápida en Colab T4 con menor consumo de VRAM.
- **Embeddings:** `intfloat/multilingual-e5-base` — Versión base del modelo de embeddings multilingüe (768 dims); menor huella de memoria que `e5-large` manteniendo buen rendimiento semántico en inglés.
- **Vector Store:** FAISS — Índice vectorial local que permite búsqueda eficiente por similitud sobre los fragmentos de Wikipedia indexados.
- **Framework:** LangChain + Gradio — LangChain orquesta la cadena RAG (recuperación → aumentación → generación), mientras Gradio expone una interfaz conversacional interactiva para el usuario.

Enfoque propuesto
---

La solución estará compuesta por los siguientes elementos:

1. **Construcción del corpus documental** sobre guerras mundiales.
2. **Preprocesamiento y fragmentación del texto** en unidades manejables.
3. **Representación semántica mediante embeddings**.
4. **Indexación vectorial** para búsqueda eficiente.
5. **Recuperación de contexto relevante** ante cada consulta.
6. **Generación de respuestas con un modelo de lenguaje**.
7. **Incorporación de historial conversacional**, de modo que el sistema pueda responder preguntas de seguimiento con referencias implícitas.
8. **Despliegue en una interfaz interactiva con Gradio**.

## Paquetes y Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore")

# Verificar si se está ejecutando en Google Colab
try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Ejecutando en Colab: {IN_COLAB}")

In [ ]:
# Instalamos las dependencias según el entorno de ejecución
# En Colab usamos FAISS con GPU; en local, la variante CPU por compatibilidad
if IN_COLAB:
    !pip install -q \
        langchain-classic langchain-ollama langchain-community \
        langchain-huggingface langchain-text-splitters \
        faiss-gpu-cu12 sentence-transformers \
        datasets gradio wordcloud \
        matplotlib seaborn nltk colab-xterm
else:
    !pip install -r requirements.txt

In [ ]:
# Importamos las librerías necesarias para las diferentes secciones
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from wordcloud import WordCloud, STOPWORDS
import nltk
import random

# Langchain, HF and LLM imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Especificos para la cadena de RAG
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

# especificos para la cadena de RAG con historial
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
# Huggingface access token (to improve the performance od dataset load)
## RUN ONLY IF YOU HAVE AN ACCESS TOKEN, IF NOT, SKIP IT


from getpass import getpass
from huggingface_hub import login

# token = getpass("Enter HF token: ")
# login(token)

#

---
# 1. Carga del Dataset

En esta fase se construye el corpus documental que servirá como base de conocimiento del sistema RAG. Dado que el chatbot estará orientado al dominio de las guerras mundiales, se seleccionan artículos relevantes que cubren eventos, actores, causas, consecuencias y elementos históricos clave.

Usamos **`wikimedia/wikipedia`** (dump 20231101 en inglés) cargado en modo **streaming** para evitar descargar el dataset completo (~20 GB).

Palabras clave de filtrado
----


In [ ]:
from datasets import load_dataset
import pandas as pd

# Palabras clave que nos ayudaran a definir nuestro corpus de Guerras Mundiales a partir de los títulos de los artículos de Wikipedia.
# Se han seleccionado términos que cubren ambos conflictos mundiales, así como aspectos generales relacionados con las guerras.
WW_KEYWORDS = [
    # General
    "World War",
    "World War I",
    "World War II",
    "First World War",
    "Second World War",
    "Great War",
    "Interwar period",
    "Allied Powers",
    "Central Powers",
    "Axis powers",
    "Triple Entente",
    "League of Nations",
    "armistice",
    # WWI específico
    "Western Front",
    "Eastern Front",
    "Trench warfare",
    "Treaty of Versailles",
    "Battle of the Somme",
    "Somme",
    "Battle of Verdun",
    "Verdun",
    "Gallipoli",
    "Passchendaele",
    "Marne",
    "Jutland",
    "Austro-Hungarian",
    "Ottoman Empire",
    "Archduke Franz Ferdinand",
    "Sarajevo assassination",
    "Black Hand",
    "Schlieffen Plan",
    "U-boat",
    "submarine warfare",
    "chemical warfare",
    "mustard gas",
    "no man's land",
    # WWII específico
    "Blitzkrieg",
    "Holocaust",
    "D-Day",
    "Normandy",
    "Nazi",
    "Third Reich",
    "Wehrmacht",
    "Luftwaffe",
    "RAF",
    "Gestapo",
    "SS",
    "Pearl Harbor",
    "Hiroshima",
    "Nagasaki",
    "Stalingrad",
    "Operation Overlord",
    "Operation Barbarossa",
    "Battle of Berlin",
    "Battle of Kursk",
    "Battle of Moscow",
    "Dunkirk",
    "El Alamein",
    "Pacific War",
    "Vichy France",
    "Manhattan Project",
    "Auschwitz",
    "Yalta Conference",
    "Potsdam Conference",
    # Batallas (ambas guerras)
    "Battle of Britain",
    "Battle of Midway",
    "Battle of Stalingrad",
    "Battle of the Bulge",
    # Tecnología y armamento
    "tank warfare",
    "Panzer",
    "air superiority",
    "strategic bombing",
    "fighter aircraft",
    "bomber aircraft",
    "radar",
    "codebreaking",
    "Enigma",
    # Consecuencias
    "Cold War",
    "United Nations",
    "post-war reconstruction",
    "war crimes",
    "Nuremberg Trials",
]

Construccion del corpus
----

Solo se retienen artículos cuyos títulos contienen palabras clave relacionadas con
la Primera o Segunda Guerra Mundial, y se trunca cada artículo a `MAX_CHARS`
caracteres para mantener tiempos de embedding razonables. El resultado es una
colección de documentos históricos verificables que servirá como base de
conocimiento del chatbot.

Se organizan los articulos en un formato utilizable para LangChain.

In [ ]:
# SE TOMA LA DECISIÓN DE UTILIZAR UNA CANTIDAD MODERADA DE ARTICULOS PARA NO SOBREDIMENSIONAR LA SOLUCIÓN
# Pero en este punto se podrian obtener miles de articulos si se requiere
MAX_ARTICLES = 1250

# POR LA MISMA RAZON, SE TRUNCA EL NUMERO MAXIMO DE CARACTERES DE LOS ARTICULOS
MAX_CHARS = 15000


# Función para verificar si el título de un artículo contiene alguna de las palabras clave (case-insensitive)
def match_keyword(title, keywords):
    title = title.lower()
    for kw in keywords:
        pattern = r"\b" + re.escape(kw.lower()) + r"\b"
        if re.search(pattern, title):
            return True
    return False


# Cargamos el dataset de Wikipedia en modo streaming para procesar los artículos uno por uno sin cargar todo el dataset en memoria
print("Cargando dataset Wikipedia en modo streaming...")
wiki_stream = load_dataset(
    "wikimedia/wikipedia", "20231101.en", split="train", streaming=True
)

# Filtramos artículos relevantes por título usando las palabras clave definidas
articles = []
for example in wiki_stream:
    title = example["title"]

    # Filtramos por keywords en el título (case-insensitive)
    if match_keyword(title, WW_KEYWORDS):

        # Limitamos el numero de caracteres maximo en los articulos
        full_text = title + "\n\n" + example["text"]
        truncated_text = full_text[:MAX_CHARS]

        # Añadimos el articulo a nuestra lista de articulos recuperados
        articles.append(
            {
                "title": title,
                "url": example["url"],
                "text": truncated_text,  # título + cuerpo truncado
            }
        )

    if len(articles) >= MAX_ARTICLES:
        break

print(f"Artículos recuperados: {len(articles)}")
print("Muestra de títulos:")
for a in random.sample(articles, 10):
    print(f'  - {a["title"]}')

In [ ]:
# Convertimos a DataFrame para facilitar la exploración
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(articles)

# Calculamos longitud en palabras de cada artículo
df["word_count"] = df["text"].apply(lambda t: len(t.split()))
df["char_count"] = df["text"].apply(len)

print(f"Shape del DataFrame: {df.shape}")
df[["title", "word_count", "char_count"]].head(25)

---
# 2. Análisis Exploratorio del Corpus (EDA)

En esta etapa se realiza un análisis detallado del corpus con el **propósito de fundamentar decisiones clave en el diseño del sistema RAG**.

A diferencia del análisis exploratorio tradicional en machine learning, aquí el enfoque está en el análisis del texto, lo cual permite tomar decisiones clave para el diseño del sistema RAG, tales como:

- El tamaño de los fragmentos (chunking),
- La selección del modelo de embeddings,
- La estrategia de recuperación de información.

---

En particular, se busca comprender la longitud y estructura de los artículos, lo cual permite definir un `chunk_size` adecuado para la segmentación del texto. Asimismo, se analiza la distribución temática del corpus para verificar el balance entre eventos de la Primera y Segunda Guerra Mundial, evitando sesgos en el proceso de recuperación.

Adicionalmente, se examinan los conceptos y términos predominantes con el fin de validar la relevancia semántica del corpus respecto al dominio de interés. Este análisis también permite identificar posibles vacíos o sobre-representaciones en la información disponible.

Estas decisiones son fundamentales para asegurar que el sistema de recuperación y generación opere de manera eficiente, maximizando la calidad del contexto proporcionado al modelo y, en consecuencia, la precisión de las respuestas generadas.

## EDA 1: Distribución de longitud de artículos

In [ ]:
sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de palabras
axes[0].hist(
    df["word_count"], bins=50, color="steelblue", edgecolor="white", alpha=0.85
)
axes[0].axvline(
    df["word_count"].mean(),
    color="red",
    linestyle="--",
    label=f'Media: {df["word_count"].mean():.0f}',
)
axes[0].axvline(
    df["word_count"].median(),
    color="orange",
    linestyle="--",
    label=f'Mediana: {df["word_count"].median():.0f}',
)
axes[0].set_title("Distribución de palabras por artículo")
axes[0].set_xlabel("Número de palabras")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Boxplot de caracteres
axes[1].boxplot(
    df["char_count"],
    vert=True,
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6),
)
axes[1].set_title("Distribución de caracteres por artículo (boxplot)")
axes[1].set_ylabel("Número de caracteres")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

# Estadísticas descriptivas clave
print("=== Estadísticas de longitud (palabras) ===")
print(f'  Media:       {df["word_count"].mean():.0f} palabras')
print(f'  Mediana:     {df["word_count"].median():.0f} palabras')
print(f'  Percentil 5: {df["word_count"].quantile(0.05):.0f} palabras')
print(f'  Percentil 25: {df["word_count"].quantile(0.25):.0f} palabras')
print(f'  Percentil 75: {df["word_count"].quantile(0.75):.0f} palabras')
print(f'  Percentil 95: {df["word_count"].quantile(0.95):.0f} palabras')
print(f'  Máximo:      {df["word_count"].max():.0f} palabras')

**Notas — Distribución de longitudes:**

La distribución es **asimétrica a la derecha**: la mayoría de artículos son cortos-medianos, pero existen artículos algo extensos que tienden a sesgar la media hacia arriba.

El histograma confirma que la mayor densidad se concentra en artículos por debajo de ~1000 palabras, mientras que el boxplot de caracteres evidencia una concentración de caracteres por debajo de la mitad del rango maximo permitido por MAX_CHAR. Esto sugiere una alta heterogeneidad en la longitud del corpus, típica de articulos Wikipedia.

**Implicación para el chunking:**

- Artículos cortos (≈ <250 palabras): existe riesgo de **sobre-fragmentación** si el `chunk_size` es muy pequeño, lo que puede romper coherencia semántica y afectar la calidad del embedding.
  
- Artículos medianos (≈ 250-2000 palabras): representan la mayoría del corpus, por lo que el `chunk_size` debe optimizarse principalmente para este rango.

- Artículos largos (>2000 palabras): requieren segmentación efectiva; chunks demasiado grandes pueden **mezclar múltiples subtemas**, reduciendo la precisión del retriever.


## EDA 2: Distribución por sub-tema (WWI, WWII, General)


In [ ]:
# Keywords especificos por guerra

WWI_KEYWORDS = [
    "Western Front",
    "Eastern Front",
    "Trench warfare",
    "Treaty of Versailles",
    "Somme",
    "Verdun",
    "Austro-Hungarian",
    "Ottoman Empire",
    "Archduke Franz Ferdinand",
    "Sarajevo assassination",
    "U-boat",
    "submarine warfare",
    "chemical warfare",
    "mustard gas",
]

WWII_KEYWORDS = [
    "Blitzkrieg",
    "Holocaust",
    "D-Day",
    "Normandy",
    "Nazi",
    "Wehrmacht",
    "Luftwaffe",
    "RAF",
    "Gestapo",
    "Pearl Harbor",
    "Hiroshima",
    "Stalingrad",
    "Operation Overlord",
    "Battle of Berlin",
]


def keyword_match(text, keywords):
    text = text.lower()
    for kw in keywords:
        pattern = r"\b" + re.escape(kw.lower()) + r"\b"
        if re.search(pattern, text):
            return True
    return False


def classify_article(title):
    title_lower = title.lower()

    is_ww1 = keyword_match(title_lower, WWI_KEYWORDS)
    is_ww2 = keyword_match(title_lower, WWII_KEYWORDS)

    if is_ww1 and is_ww2:
        return "Ambas/General"
    elif is_ww1:
        return "WWI"
    elif is_ww2:
        return "WWII"
    else:
        return "General"


# Clasificación
df["theme"] = df["title"].apply(classify_article)

# Conteo
theme_counts = df["theme"].value_counts()

# Visualización
fig, ax = plt.subplots(figsize=(7, 7))

colors = {
    "WWI": "#4472C4",
    "WWII": "#ED7D31",
    "Ambas/General": "#A9D18E",
    "General": "#BFBFBF",
}

ax.pie(
    theme_counts.values,
    labels=theme_counts.index,
    autopct="%1.1f%%",
    colors=[colors.get(k, "#CCCCCC") for k in theme_counts.index],
    startangle=140,
    textprops={"fontsize": 12},
)

ax.set_title("Distribución del corpus por sub-tema", fontsize=14, pad=15)

plt.tight_layout()
plt.show()

# Output textual
print("Conteo por sub-tema:")
print(theme_counts.to_string())

**Notas - Implicaciones de deistribucion de temas**

El gráfico revela un desbalance significativo entre las categorías:  

- **General:** La mayoría de los artículos recuperados no tienen palabras clave exclusivas de WWI o WWII en el título, pero sí son relevantes temáticamente (batallas sin nombre de guerra, figuras militares, armamento, geografía de conflictos, etc.). Esto indica que el filtrado por título es conservador: captura contexto bélico amplio más allá de las etiquetas explícitas.

- **WWII**: La Segunda Guerra Mundial tiene una cobertura significativamente mayor en Wikipedia en inglés. Eventos como el Holocausto, el Día D y Stalingrad generan muchos más artículos derivados que sus equivalentes de WWI.

- **WWI**: La Primera Guerra Mundial está notablemente subrepresentada. Sus keywords son menos frecuentes en títulos de Wikipedia — gran parte del contenido de WWI está integrado en artículos generales sobre el período 1914–1918.

## EDA 3: WordCloud del contenido

In [ ]:
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

# Tomamos las primeras 500 palabras de cada artículo para acelerar el proceso
# Usamos stopwords de NLTK para filtrar términos vacíos

en_stopwords = set(nltk_stopwords.words("english")) | STOPWORDS

# Concatenamos extractos de todos los artículos
content_sample = " ".join(
    " ".join(text.split()[:500])  # primeras 500 palabras de cada artículo
    for text in df["text"].tolist()
)

wc_content = WordCloud(
    width=900,
    height=450,
    background_color="#1a1a2e",
    stopwords=en_stopwords,
    colormap="YlOrRd",
    max_words=80,
    collocations=False,
).generate(content_sample)

# Visualización del WordCloud de contenido
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc_content, interpolation="bilinear")
ax.axis("off")
ax.set_title("WordCloud — Conceptos dominantes en el contenido del corpus", fontsize=14)
plt.tight_layout()
plt.show()

## EDA 4: Ranking palabras en el contenido

In [ ]:
from collections import Counter

# Limpieza básica de palabras: minúsculas, eliminación de puntuación y stopwords
words_clean = [word.lower().strip(".,;:!?()[]\"'") for word in content_sample.split()]

# Filtramos stopwords y palabras muy cortas
words_clean = [
    word for word in words_clean if word not in en_stopwords and len(word) > 2
]

# Contamos las palabras limpias y visualizamos las 20 más comunes
common_words_clean = Counter(words_clean).most_common(20)
df_words_clean = pd.DataFrame(common_words_clean, columns=["word", "freq"])

plt.figure(figsize=(10, 5))
plt.bar(df_words_clean["word"], df_words_clean["freq"])
plt.xticks(rotation=45)
plt.title("Top 20 palabras más frecuentes (sin stopwords)")
plt.xlabel("Palabra")
plt.ylabel("Frecuencia")
plt.show()

Notas generales del EDA
----

El análisis exploratorio del corpus evidencia varios patrones relevantes que impactan directamente el diseño del sistema RAG.

En primer lugar, se observa un **desbalance temático significativo**, con una clara predominancia de artículos generales y, en menor medida, de la Segunda Guerra Mundial, mientras que la Primera Guerra Mundial está subrepresentada. Esto sugiere que el sistema de recuperación tenderá a responder con mayor precisión en consultas relacionadas con WWII o temas generales, lo cual debe considerarse como una limitación inherente del corpus.

En términos de longitud, los artículos presentan una **alta variabilidad**. Esto implica que el tamaño de los chunks debe balancear adecuadamente **coherencia semántica y granularidad**, evitando tanto la fragmentación excesiva como la mezcla de múltiples subtemas en un mismo segmento.

El análisis léxico refuerza la validez del corpus: los términos dominantes en el contenido (e.g., *Force, RAF, War, forces, attack, Army*) indican una fuerte presencia de lenguaje militar y contextual relevante.

En conjunto, el corpus es suficientemente rico en semántica de dominio, pero presenta **desbalance y ruido estructural**, lo cual puede afectar la precisión del retriever en ciertos casos.

---

### Implicación para el chunking

Dado este comportamiento, se plantea como hipótesis inicial evaluar tres configuraciones de tamaño de chunk:

- `256` caracteres: mayor granularidad, pero riesgo de pérdida de contexto  
- `512` caracteres: equilibrio entre coherencia y precisión  
- `800` caracteres: mayor contexto, pero posible dilución semántica  

Estas configuraciones serán evaluadas en la siguiente sección para determinar cuál optimiza la calidad de recuperación en el sistema.

---
# 3. Preprocesamiento y Análisis de Sensibilidad al Chunking

Los modelos de embedding tienen una ventana de tokens limitada (típicamente 512 tokens). Los artículos de Wikipedia suelen superar ese límite ampliamente, por lo que **debemos dividirlos en fragmentos (chunks)** antes de indexarlos en el vector store.

No existe un valor universal de `chunk_size` óptimo — depende del corpus, del modelo de embedding y del tipo de preguntas que se espera recibir. Por eso realizamos un **análisis de sensibilidad**: evaluamos tres configuraciones (pequeña, mediana y grande) y comparamos cómo afectan la distribución y la cantidad de chunks generados.

El objetivo es tomar una decisión **informada y justificada** sobre qué configuración usar en el RAG, en lugar de elegir un valor arbitrario. Los resultados del EDA (longitud promedio de los artículos) sirven como punto de partida para acotar el rango de valores a explorar.

**`RecursiveCharacterTextSplitter`**


LangChain ofrece este splitter que intenta dividir respetando la estructura natural del texto: primero por párrafos (`\n\n`), luego por líneas (`\n`), y finalmente por espacios. Es el más robusto para texto narrativo como Wikipedia porque preserva la coherencia semántica dentro de cada fragmento.


**Parámetros clave**

- **`chunk_size`**: número máximo de caracteres por fragmento. 
- **`chunk_overlap`**: solapamiento entre fragmentos consecutivos. Evita cortar frases a la mitad y preserva la continuidad temática entre chunks adyacentes.



In [ ]:
# Convertimos cada artículo a un objeto Document de LangChain
# Los metadatos (título, URL) se conservarán en cada chunk para citar la fuente
docs = [
    Document(page_content=a["text"], metadata={"title": a["title"], "url": a["url"]})
    for a in articles
]

print(f"Documentos creados: {len(docs)}")
print(f"Ejemplo de metadata: {docs[0].metadata}")
print(f"Longitud del primer documento: {len(docs[0].page_content)} chars")

## Prueba de Sensibilidad al Chunking

In [ ]:
# Definimos 3 configuraciones que abarcan el espacio de decisión relevante

# Notese que estamos utilizando un chunk overlap relativamente pequeño
# con la intención de simplificar el dimensionamiento de nuestro Vector database

CHUNK_CONFIGS = {
    "small": {"chunk_size": 256, "chunk_overlap": 16},  # fragmento corto
    "medium": {"chunk_size": 512, "chunk_overlap": 32},  # ~1 párrafo
    "large": {"chunk_size": 800, "chunk_overlap": 64},  # ~1-3 párrafos
}

split_results = {}
for name, cfg in CHUNK_CONFIGS.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        # Separadores: párrafos - líneas - espacios - caracteres
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    lengths = [len(c.page_content) for c in chunks]
    split_results[name] = {
        "chunks": chunks,
        "lengths": lengths,
        "n": len(chunks),
        "avg": int(sum(lengths) / len(lengths)),
        "median": int(sorted(lengths)[len(lengths) // 2]),
        "min": min(lengths),
        "max": max(lengths),
    }

# Tabla resumen
print(
    f"{'Config':<10} {'N° chunks':>10} {'Avg (chars)':>12} {'Median':>8} {'Min':>6} {'Max':>8}"
)
print("-" * 58)
for name, r in split_results.items():
    print(
        f'{name:<10} {r["n"]:>10,} {r["avg"]:>12,} {r["median"]:>8,} {r["min"]:>6,} {r["max"]:>8,}'
    )

In [ ]:
#  Visualización de las distribuciones de longitud por configuración

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
palette = {"small": "#4472C4", "medium": "#ED7D31", "large": "#70AD47"}

for ax, (name, r) in zip(axes, split_results.items()):
    ax.hist(r["lengths"], bins=40, color=palette[name], edgecolor="white", alpha=0.85)
    ax.axvline(
        r["avg"], color="red", linestyle="--", linewidth=1.5, label=f'Media: {r["avg"]}'
    )
    ax.axvline(
        r["median"],
        color="black",
        linestyle=":",
        linewidth=1.5,
        label=f'Mediana: {r["median"]}',
    )
    ax.set_title(
        f'chunk_size={CHUNK_CONFIGS[name]["chunk_size"]} ({name})\nN={r["n"]:,} chunks',
        fontsize=12,
    )
    ax.set_xlabel("Longitud del chunk (chars)")
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=9)

plt.suptitle(
    "Distribución de longitud de chunks por configuración", fontsize=14, y=1.02
)
plt.tight_layout()
plt.show()

## Decisión de Chunking

A partir del EDA y el análisis de sensibilidad, se observa un comportamiento consistente en las tres configuraciones: la distribución de longitudes va en incremento hacia el límite superior del `chunk_size`, lo que indica que el splitter aprovecha al máximo la capacidad definida antes de fragmentar. Esto es deseable, pues sugiere que los chunks contienen unidades semánticas relativamente completas en lugar de cortes prematuros.

| Config | Pros | Contras |
|---|---|---|
| **small** (256/16) | Alta granularidad, mayor precisión en hechos puntuales | Fragmentación excesiva, pérdida de coherencia semántica |
| **medium** (512/32) | Balance adecuado entre contexto y precisión | Ligera redundancia por overlap |
| **large** (800/64) | Mayor contexto por chunk, útil para consultas complejas | Riesgo de mezclar múltiples subtemas y reducir precisión del retriever |

Adicionalmente, los histogramas permiten observar cualitativamente que:

- En **small**, la mediana se sitúa considerablemente por debajo del límite, lo que sugiere que muchos chunks quedan incompletos semánticamente — probablemente párrafos cortos o listas de Wikipedia que no alcanzan el tamaño máximo.
- En **medium**, la mediana se desplaza hacia el centro-superior del rango, indicando que la mayoría de los chunks capturan unidades textuales más coherentes y completas.
- En **large**, aunque se maximiza el contexto disponible por chunk, la distribución evidencia que muchos fragmentos alcanzan el tope del límite, lo que aumenta la probabilidad de introducir ruido semántico al combinar múltiples subtemas dentro de un mismo chunk.

---

### Elección final

Se selecciona **`medium` (512 chars / 32 overlap)** como configuración principal, ya que ofrece el mejor compromiso entre granularidad y coherencia semántica.

Esta decisión se fundamenta en:

1. La estructura típica de los artículos de Wikipedia, donde los párrafos suelen concentrar información semánticamente consistente dentro de rangos intermedios de longitud.
2. La distribución observada en el análisis de sensibilidad, donde esta configuración evita tanto la sobre-fragmentación como la sobre-agregación de contenido.
3. La compatibilidad con el modelo de embeddings (`multilingual-e5-base`), cuya ventana de entrada se ajusta cómodamente a fragmentos de este tamaño sin riesgo de truncamiento.
4. El uso de un `overlap` de 32 caracteres, que introduce redundancia controlada y mejora la continuidad entre fragmentos consecutivos, favoreciendo la recuperación de contexto relevante.

En conjunto, esta configuración maximiza la probabilidad de que cada chunk represente una unidad semántica coherente, lo cual es crítico para el desempeño del retriever en tareas de búsqueda y generación aumentada.

> La validación final se realizará de forma cualitativa en el sistema RAG, evaluando si los chunks recuperados mantienen coherencia y relevancia frente a distintas consultas.

In [ ]:
# Seleccionamos la configuración medium como la principal para el RAG
chunks = split_results["medium"]["chunks"]
print(f"Chunks seleccionados (medium): {len(chunks):,}")
print(f"Ejemplo de chunk:")
print(f"  Metadata: {chunks[0].metadata}")
print(f"  Contenido (primeros 300 chars): {chunks[0].page_content[:300]}...")

---
# 4. Configuración del LLM: phi3:mini

| Característica | llama3.2:3b | mistral:7b-instruct | **phi3:mini** |
|---|---|---|---|
| Parámetros | 3B | 7B | **3.8B** |
| Ventana de contexto | 128K | 32K | **128K** |
| Benchmark MMLU | ~58% | ~64% | **~68%** |
| Instruction-following | Bueno | Muy bueno | **Muy bueno** |
| Uso de VRAM (cuantizado) | ~2 GB | ~4.1 GB | **~2.3 GB** |

Se elige **`phi3:mini`** de Microsoft como modelo de generación. A pesar de ser un modelo compacto (~3.8B parámetros), supera en benchmarks de razonamiento a alternativas de tamaño similar y es comparable con modelos más grandes gracias a su entrenamiento orientado a instrucciones. Su bajo consumo de VRAM lo hace ideal para ejecutarse en Colab T4 sin comprometer la calidad ni latencia de las respuestas históricas.

> **Nota:** Necesitamos que `ollama serve` esté corriendo antes de ejecutar las siguientes celdas.

In [ ]:
# Instalamos dependencias del sistema y Ollama si no está disponible
!sudo apt install zstd -y -q
!if ! type ollama > /dev/null 2>&1; then \
    echo 'Instalando Ollama...' && curl -fsSL https://ollama.com/install.sh | sh; \
else \
    echo 'Ollama ya está instalado.'; \
fi

In [ ]:
# Cargamos la extensión de terminal para Colab
# En la terminal que se abre, ejecuta: ollama serve &
# Esto lanza el servidor de Ollama en segundo plano
%load_ext colabxterm
%xterm

In [ ]:
# Descargamos el modelo phi3:mini desde el repositorio de Ollama
!ollama pull phi3:mini

## Sanity check

In [ ]:
import time

# Initialize the LLM
# temperature=0.25: casi determinista para precisión histórica
llm = ChatOllama(model="phi3:mini", temperature=0.25)

print("Verificando conexión con Ollama...")
# Damos un pequeño margen por si el servidor se acaba de lanzar
time.sleep(2)

try:
    # Prueba de sanity: verificamos que el LLM responde
    response = llm.invoke("Who started World War I and what were the main causes?")
    print("=== Respuesta del LLM (sin contexto RAG) ===")
    print(response.content)
except Exception as e:
    print(f"Error de conexión: {e}")
    print("\nASEGÚRATE DE QUE:")
    print("1. Ejecutaste 'ollama serve &' en la xterm del paso anterior.")
    print("2. El modelo 'phi3:mini' se descargó correctamente con 'ollama pull'.")

----

# 5. Creación del Vector Store con FAISS

En esta fase se transforman los fragmentos del corpus en representaciones vectoriales mediante embeddings. Esta representación permite que el sistema compare consultas y documentos en un espacio semántico, en lugar de depender únicamente de coincidencias literales de palabras.

Posteriormente, estos vectores se almacenan en una base vectorial utilizando FAISS, lo que hace posible recuperar de manera eficiente los fragmentos más relevantes frente a una pregunta del usuario.

Pipeline de indexación
---

```
chunks (texto) → Embedding Model → vectores float32 → FAISS Index
```

1. **`intfloat/multilingual-e5-base`**: modelo de embedding entrenado en 94 idiomas. Produce vectores de 768 dimensiones. Es la versión base de la familia e5 — menor huella de memoria que `e5-large` manteniendo buen rendimiento semántico en inglés (MTEB benchmark).

2. **FAISS** (Facebook AI Similarity Search): librería especializada en búsqueda de vecinos más cercanos en espacios vectoriales de alta dimensión. Usa índices aproximados para escalar a millones de vectores manteniendo baja latencia.

3. **Guardado del índice**: el índice FAISS se serializa a disco. En ejecuciones posteriores se carga directamente, evitando recalcular todos los embeddings.

In [ ]:
import os

# Inicializamos el modelo de embeddings con normalización L2
# necesario para similitud coseno correcta
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    encode_kwargs={"normalize_embeddings": True},
)

index_path = "./faiss_ww_index"

if os.path.exists(index_path):
    # Cargamos el índice existente (evita re-embeddear todo el corpus)
    print("Cargando índice FAISS existente...")
    vectorstore = FAISS.load_local(
        index_path, embeddings, allow_dangerous_deserialization=True
    )
else:
    # Construimos el índice desde los chunks -  PUEDE TARDAR UNOS MINUTOS
    print(f"Construyendo índice FAISS para {len(chunks):,} chunks...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(index_path)
    print("Índice guardado en disco.")

Creacion del retriever
---

Se define un mecanismo de recuperación que seleccionará los fragmentos más relevantes para cada consulta. En esta etapa inicial se utilizará `k=4`, es decir, se recuperarán los cuatro chunks más similares semánticamente a la pregunta del usuario.

Este valor busca equilibrio entre cobertura contextual y reducción de ruido.

In [ ]:
# El retriever buscará los 4 chunks más similares a cada consulta (k=4)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Retriever listo. Índice contiene {vectorstore.index.ntotal:,} vectores.")

Pruebas del retriever
---

In [ ]:
test_queries = [
    "Who was Adolf Hitler?",
    "What was the Treaty of Versailles?",
    "What happened on D-Day?",
    "What caused World War II?",
]

for q in test_queries:
    docs = retriever.invoke(q)
    print("=" * 100)
    print("QUERY:", q)
    print("TOP RESULT TITLE:", docs[0].metadata.get("title", "Sin título"))
    print("TOP RESULT PREVIEW:")
    print(docs[0].page_content[:300])
    print("\n")

**Notas del retriever**

La validación del retriever muestra resultados satisfactorios. Para consultas directas sobre entidades, tratados y eventos específicos, el sistema recupera como primer resultado el documento más relevante del corpus, como se observó en los casos de Adolf Hitler, Treaty of Versailles y D-Day.

En conjunto, los resultados evidencian que la base vectorial y el retriever están funcionando correctamente y ofrecen una base sólida para integrar el modelo generativo en la siguiente fase.

---
# 6. Pipeline RAG: Pregunta-Respuesta Simple



En esta fase se integra el modelo de lenguaje con el componente de recuperación semántica construido en etapas anteriores. El objetivo es que el sistema no responda únicamente a partir del conocimiento paramétrico del modelo, sino apoyándose en fragmentos relevantes del corpus documental.

De esta manera, el flujo general del sistema será el siguiente:

1. El usuario formula una pregunta.
2. El retriever busca los fragmentos más relevantes en la base vectorial.
3. Estos fragmentos se incorporan como contexto en un prompt.
4. El modelo de lenguaje genera una respuesta fundamentada en la información recuperada.

Usamos `create_retrieval_chain` (forma moderna, no deprecada) que devuelve automáticamente el contexto recuperado como lista de `Document` objects, facilitando la cita de fuentes.


In [ ]:
# Prompt en inglés para alinearlo con el corpus
# Instruimos al LLM a citar fuentes y admitir incertidumbre
prompt = PromptTemplate.from_template(
    """\
You are a knowledgeable historian specializing in World War I and World War II.
Use the following context fragments retrieved from Wikipedia to answer the question.
If the answer is not in the context, say so honestly — do not hallucinate.
Always include citations to the source articles at the end of your response.

Context:
{context}

Question: {input}
Answer:"""
)

# create_stuff_documents_chain concatena todos los chunks en el campo {context}
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# create_retrieval_chain une el retriever con la cadena de documentos
qa_chain = create_retrieval_chain(retriever, combine_docs_chain)

Desarrollamos una función para devolver la respuesta de las preguntas del usuario de una forma mas clara, mostrando las fuentes usadas y sin duplicar titulos debido a chunks del mismo articulo.

In [ ]:
# Funcion para formatear la respuesta incluyendo título y URL de cada fuente recuperada
# Evitamos repetir fuentes con el mismo título (aunque tengan URLs distintas) para mayor claridad
def format_answer(result):
    """Formatea la respuesta incluyendo título y URL de cada fuente recuperada."""
    answer = result["answer"] + "\n\n**Fuentes:**\n"
    seen = set()
    for i, doc in enumerate(result["context"], 1):
        title = doc.metadata.get("title", "Desconocido")
        url = doc.metadata.get("url", "")
        key = title  # deduplicamos por título
        if key not in seen:
            answer += f"  [{i}] {title} — {url}\n"
            seen.add(key)
    return answer


print("Cadena RAG configurada correctamente.")

## Pruebas con preguntas complejas

In [ ]:
# Pruebas con preguntas históricas
test_questions = [
    "What were the main causes of World War I?",
    "Describe the D-Day invasion and its significance.",
    "Who was Winston Churchill and what role did he play in World War II?",
    "What was the impact of the Treaty of Versailles on post-war Europe?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {q}")
    print("=" * 60)
    result = qa_chain.invoke({"input": q})
    print(format_answer(result))

Observaciones sobre la calidad del RAG simple
---

Los resultados obtenidos evidencian que el sistema RAG funciona de manera satisfactoria en consultas tanto específicas como conceptuales.

En preguntas sobre entidades y eventos concretos el sistema recupera información pertinente y genera respuestas claras, coherentes y alineadas con el corpus documental.

En preguntas más amplias, como las causas de la Primera Guerra Mundial, el sistema también ofrece respuestas razonables y fundamentadas en las fuentes recuperadas. No obstante, se observa que en algunos casos aparece cierto ruido semántico, ya que entre las fuentes recuperadas pueden incluirse documentos temáticamente cercanos pero no estrictamente necesarios, como ocurrió con un fragmento relacionado con las causas de la Segunda Guerra Mundial.

Aun así, este comportamiento no impide que la respuesta final sea adecuada. En conjunto, los resultados confirman que la arquitectura implementada integra correctamente recuperación documental y generación de lenguaje natural.

- **Precisión de retrieval**: Las fuentes recuperadas deben corresponder temáticamente a la pregunta. Si aparecen artículos irrelevantes, podría indicar que `k=4` es demasiado alto o que el corpus tiene ruido.

- **Limitación del RAG simple**: cada pregunta es independiente — si haces preguntas de seguimiento ('¿Y el general que lo lideró?'), el modelo no tiene memoria de la pregunta anterior. Esto lo solucionamos en la siguiente sección.

---
# 7. Cadena Conversacional con Historial


En esta fase se incorpora memoria conversacional al sistema RAG con el propósito de permitir preguntas de seguimiento y referencias implícitas entre turnos de diálogo.

Hasta este punto, el sistema era capaz de responder correctamente a preguntas individuales apoyándose en el corpus documental recuperado. Sin embargo, para lograr una interacción más natural, es necesario que el chatbot conserve información de intercambios previos y la utilice como contexto adicional en nuevas consultas.

Objetivos de la fase
---

- Almacenar el historial de la conversación
- Incorporar dicho historial al prompt del sistema
- Permitir referencias implícitas a entidades o eventos mencionados previamente
- Evaluar si el chatbot responde con coherencia entre varios turnos consecutivos

--

**`create_history_aware_retriever`**

Para abordar la limitación de un RAG simple, se introduce un retriever consciente del historial que incorpora una etapa previa de reformulación. En este enfoque, el modelo utiliza el historial conversacional junto con la nueva pregunta para generar una versión explícita y autónoma de la consulta.

El flujo es el siguiente:

- historial + nueva pregunta  
 
- el LLM reformula la pregunta (ej. *"Who commanded the German forces at Stalingrad?"*)  
 
- retriever  
  
- LLM genera respuesta  

De esta forma, el sistema transforma preguntas ambiguas en consultas completas antes de la recuperación, mejorando significativamente la relevancia de los documentos recuperados y la coherencia de las respuestas en contextos conversacionales.

In [ ]:
# Prompt para condensar el historial + nueva pregunta en una pregunta autónoma
condense_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given the conversation history and the latest user question, "
            "reformulate the question into a standalone question that can be understood "
            "without the conversation context. Keep the original language. "
            "Do NOT answer the question — only reformulate it if needed.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

# El retriever ahora recibe la pregunta condensada
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, condense_prompt
)

# Prompt principal del QA conversacional
qa_system_prompt = (
    "You are a knowledgeable historian specializing in World War I and World War II. "
    "Use the following retrieved context to answer the question. "
    "If you cannot find the answer in the context, say so. "
    "Be concise but informative.\n\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

# Cadena final: retriever consciente del historial + generación
qa_chain_conv = create_stuff_documents_chain(llm, qa_prompt)
convo_qa_chain = create_retrieval_chain(history_aware_retriever, qa_chain_conv)

print("Cadena conversacional configurada.")

## Demostración de conversación multi-turno

In [ ]:
# Mostramos cómo el historial permite preguntas de seguimiento con referencias implícitas

chat_history = []


def ask_qa_chain(question, chat_history):
    """Invoca la cadena RAG, actualiza el historial e imprime la respuesta con sus fuentes."""
    print(f"\n{'─'*60}")
    print(f"Usuario: {question}")
    print("─" * 60)

    response = convo_qa_chain.invoke({"input": question, "chat_history": chat_history})

    # Actualizamos el historial para la siguiente vuelta
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response["answer"]))

    # Mostramos la respuesta y las fuentes recuperadas
    sources = list({d.metadata["title"] for d in response["context"]})
    print(f'Asistente: {response["answer"]}')
    print(f"[Fuentes: {', '.join(sources[:3])}]")


conversation = [
    "Tell me about the Battle of Stalingrad.",
    "Who commanded the German forces there?",  # 'there' = Stalingrad
    "What happened to him after the battle?",  # 'him' = von Paulus
]

for question in conversation:
    ask_qa_chain(question, chat_history)

## Prueba con eventos militares

In [ ]:
chat_history = []

q1 = "What happened on D-Day?"
r1 = ask_qa_chain(q1, chat_history)

print("Q1:", q1)
print("A1:", r1["answer"])
print("Sources:", r1["sources"])

In [ ]:
q2 = "Why was that operation significant?"
r2 = ask_qa_chain(q2, chat_history)

print("Q2:", q2)
print("A2:", r2["answer"])
print("Sources:", r2["sources"])

In [ ]:
q3 = "Which side benefited the most from it?"
r3 = ask_qa_chain(q3, chat_history)

print("Q3:", q3)
print("A3:", r3["answer"])
print("Sources:", r3["sources"])

In [ ]:
chat_history

**Notas - Chat con memoria temporal**

Los resultados obtenidos evidencian que el sistema es capaz de mantener coherencia a lo largo de múltiples turnos de conversación, interpretando correctamente referencias implícitas como pronombres ("he", "it", "that operation") y relacionándolos con entidades o eventos mencionados previamente.

En los casos evaluados, el chatbot logra:

- identificar correctamente a los sujetos de preguntas de seguimiento,
- mantener continuidad temática entre turnos,
- y generar respuestas coherentes apoyadas en el contexto recuperado.

Esto demuestra que la incorporación del historial conversacional permite transformar el sistema de un esquema de preguntas independientes a un modelo interactivo capaz de sostener diálogos contextualizados.

**Con esta mejora, el sistema evoluciona hacia un chatbot conversacional más natural, cumpliendo con los requisitos de interacción continua y contextualización entre preguntas.**

---
# 8. Interfaz de Usuario — ChatBot Gradio

En esta fase se implementa una interfaz gráfica en Gradio para interactuar con el sistema RAG de manera conversacional. Esta capa permite validar el comportamiento del chatbot en un entorno más cercano al uso real, facilitando la demostración de respuestas fundamentadas y del uso del historial conversacional.

Objetivos de la fase
---

- Conectar el pipeline RAG con una interfaz amigable
- Mostrar el historial de conversación en tiempo real
- Permitir preguntas de seguimiento con referencias implícitas
- Preparar el sistema para su demostración final


Construimos la interfaz conversacional con **Gradio Blocks**, que ofrece más flexibilidad que `gr.ChatInterface`.

Gestión dual de historiales
---

| Historial | Tipo | Propósito |
|---|---|---|
| `lc_history` | `List[HumanMessage \| AIMessage]` | Pasado a LangChain en cada invocación |
| `chat_history` (Gradio) | `List[Tuple[str, str]]` | Renderizado visual en el componente `gr.Chatbot` |

Ambos historiales se resetean al presionar el botón **Limpiar**, garantizando que una nueva conversación no herede contexto de la anterior.

In [ ]:
import gradio as gr

# Historial de LangChain (estructuras internas)
lc_history = []


def respond(question, chat_history):
    """Procesa la pregunta del usuario y actualiza ambos historiales."""
    if not question.strip():
        return "", chat_history

    # Invocamos la cadena conversacional con el historial de LangChain
    reply = convo_qa_chain.invoke({"input": question, "chat_history": lc_history})

    # Actualizamos el historial interno de LangChain
    lc_history.append(HumanMessage(content=question))
    lc_history.append(AIMessage(content=reply["answer"]))

    # Formateamos respuesta con fuentes para la UI
    answer = format_answer(reply)

    # Actualizamos el historial visual de Gradio
    chat_history.append((question, answer))
    return "", chat_history


def reset_chat():
    """Limpia ambos historiales para iniciar una conversación nueva."""
    lc_history.clear()  # vaciamos el historial de LangChain
    return [], ""  # vaciamos la UI de Gradio


# Construimos la interfaz
with gr.Blocks(title="WW Historian RAG", theme=gr.themes.Soft()) as gr_blocks:
    gr.Markdown(
        """
    # Historiador de las Guerras Mundiales — RAG
    Pregunta sobre la **Primera** o **Segunda Guerra Mundial**.
    Las respuestas están fundamentadas en artículos de Wikipedia y el modelo **phi3-mini Instruct** de Microsoft.
    """
    )

    chatbot = gr.Chatbot(label="Conversación", height=450)
    msg = gr.Textbox(
        label="Tu pregunta",
        placeholder="Ej: What were the main battles of World War II?",
        lines=2,
    )

    with gr.Row():
        submit_btn = gr.Button("Enviar", variant="primary")
        clear_btn = gr.Button("Limpiar conversación")

    # Enviamos con Enter o con el botón
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(reset_chat, None, [chatbot, msg], queue=False)

gr_blocks.launch(inline=False, share=IN_COLAB)

# Questions to Ask

# What was the Treaty of Versailles?
# Why was it important?
# Did it have consequences for Germany?

# What happened on D-Day?
# Why was that operation significant?
# Which side benefited the most from it?

In [ ]:
# Cerramos la interfaz de Gradio al finalizar el taller
gr_blocks.close()

---
# 9. Conclusiones

Resumen del taller
---

Implementamos un **sistema RAG completo** especializado en la historia de las Guerras Mundiales, recorriendo todo el pipeline desde la curaduría del corpus hasta la interfaz conversacional:

1. **Curaduría del corpus**: Filtrado semántico de `wikimedia/wikipedia` en modo streaming, reteniendo artículos temáticamente relevantes para WWI y WWII mediante palabras clave en el título. Se aplicó un límite de caracteres por artículo para mantener tiempos de embedding razonables.
2. **EDA informada**: El análisis de distribución de longitudes, composición temática (pie chart) y nubes de palabras fundamentó directamente las decisiones de chunking y nos alertó sobre el desbalance entre WWI y WWII en el corpus.
3. **Chunking con análisis de sensibilidad**: Comparamos tres configuraciones (small 256/16, medium 512/32, large 800/64) y justificamos `medium` como el balance óptimo entre granularidad semántica y contexto por fragmento.
4. **Indexación vectorial con FAISS**: Usamos `intfloat/multilingual-e5-base` para generar embeddings de 768 dimensiones y FAISS como índice vectorial eficiente. El índice se persiste en disco para evitar recalcular embeddings en ejecuciones posteriores.
5. **Cadena conversacional**: Incorporamos `create_history_aware_retriever` para reformular preguntas de seguimiento usando el historial, permitiendo conversaciones multi-turno naturales.
6. **Interfaz Gradio**: Desplegamos el chatbot con gestión correcta de los dos historiales independientes (LangChain + Gradio), con botón de reset que limpia ambos.

Tabla comparativa: configuraciones de chunking
---

| Config | N° chunks | Granularidad | Contexto por chunk | Recomendado para |
|---|---|---|---|---|
| small (256/16) | Mayor | Alta | Bajo | Preguntas de hechos muy concretos |
| **medium (512/32)** | **Medio** | **Media** | **Medio** | **Uso general (elegido)** |
| large (800/64) | Menor | Baja | Alto | Preguntas de análisis y síntesis |

Análisis crítico — Limitaciones
---

- **Desbalance del corpus**: el EDA reveló que ~60% de los artículos son de categoría *General* y solo ~5% son exclusivamente de WWI. El retriever responderá con mayor precisión preguntas sobre WWII; para WWI dependerá de contexto indirecto, lo que puede introducir ruido.
- **Capacidad del LLM**: `phi3:mini` fue elegido por su eficiencia en recursos, pero al ser un modelo de 3.8B parámetros puede mostrar limitaciones en síntesis de respuestas complejas o con múltiples fuentes. Un modelo mayor como `mistral:7b-instruct` mejoraría la calidad a costa de mayor VRAM.
- **Retriever con k=4**: reducir el número de chunks recuperados acelera la inferencia pero puede omitir información relevante para preguntas que abarcan múltiples eventos o períodos. En un escenario de producción convendría evaluar k entre 3 y 7.
- **Evaluación cualitativa**: no implementamos métricas automáticas de calidad RAG (ej. RAGAS, que mide *faithfulness*, *answer relevancy* y *context precision*). Esto es una mejora directa y prioritaria para trabajo futuro.
- **Chunking fijo**: usamos `RecursiveCharacterTextSplitter` con tamaño fijo. Enfoques más avanzados como *semantic chunking* (dividir por cambios de tema detectados con embeddings) podrían mejorar la coherencia de cada fragmento.

Posibles mejoras
---

- **Evaluación automática**: integrar RAGAS para medir la calidad de las respuestas generadas de forma sistemática.
- **Re-ranking**: aplicar un modelo cross-encoder para reordenar los chunks recuperados antes de pasarlos al LLM.
- **Ampliación del corpus**: incluir fuentes adicionales como libros de historia digitalizados o artículos académicos para reducir el desbalance WWI/WWII y enriquecer el conocimiento del sistema.